**Week 3 — Rank, Null space, Projection, Least Squares และ Condition number (ชุดฝึกธุรกิจ)**

สรุปเงื่อนไขการประมวลผลข้อมูล
การอ่านไฟล์: อ่านด้วย pd.read_csv('*.csv').to_numpy() (Pandas จะนำแถวแรกไปเป็น header อัตโนมัติ และแปลงเฉพาะตัวข้อมูลเป็น NumPy Array)

การนับดัชนี (0-based Indexing): เริ่มนับจาก 0 ทั้งดัชนีแถว คอลัมน์ ลำดับลูกค้า (biz_customers.csv) และลำดับแคมเปญ (biz_campaigns.csv)

ไฟล์ biz_customers.csv

age: อายุของลูกค้า (ปี)

income_k: รายได้ต่อเดือน (พันบาท)

tenure_months: ระยะเวลาที่เป็นลูกค้า (เดือน)

loyalty_member: เป็นสมาชิกสะสมแต้มหรือไม่ (1 = ใช่)

web_visits: จำนวนครั้งที่เข้าเว็บ/แอปใน 30 วัน

email_opens: จำนวนอีเมลโปรโมชันที่เปิดอ่าน (จาก 20 ฉบับ)

basket_size: จำนวนสินค้าเฉลี่ยต่อออเดอร์ (ขนาดตะกร้า)

spend_k: ยอดซื้อต่อเดือน (พันบาท)

ไฟล์ biz_campaigns.csv

tv: งบโฆษณาทีวี (พันบาท)

radio: งบโฆษณาวิทยุ (พันบาท)

social: งบโฆษณาโซเชียล (พันบาท)

email: งบโฆษณาอีเมล (พันบาท)

season: ไตรมาสที่จัดแคมเปญ (Q1–Q4)

sales: ยอดขายของแคมเปญ (พันชิ้น)

total_budget: งบโฆษณารวมทุกสื่อ (พันบาท)

ไฟล์ pb03_M.csv

เมทริกซ์ A ขนาด 14x18 (แถว = สูตรเมนู, คอลัมน์ = วัตถุดิบ 18 ชนิด)

โจทย์ข้อ 3.1: ไฟล์ pb03_M.csv คือตารางปริมาณวัตถุดิบ 18 ชนิดที่ใช้ในแต่ละสูตรเมนูของเครือร้านกาแฟ 14 สูตร เป็นเมทริกซ์ A ขนาด 14×18 (แถว = สูตร, คอลัมน์ = วัตถุดิบ, ไม่นับแถวชื่อคอลัมน์) Null space ของ A (เซตของเวกเตอร์ x ยาว 18 ที่ทำให้ Ax = 0) มี dimension เท่าใด = (จำนวนเต็ม)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ pb03_M.csv (เมทริกซ์ 14x18)
A = pd.read_csv("pb03_M.csv").to_numpy(dtype=np.float64)
print(f"ขนาดของ A: {A.shape}")

# 2. หา rank ของ A ด้วย np.linalg.matrix_rank
rank_A = np.linalg.matrix_rank(A)
print(f"Rank(A) = {rank_A}")

# 3. ใช้ Rank-Nullity Theorem: dim(Null space) = จำนวนคอลัมน์ - rank(A)
n_cols = A.shape[1]  # 18
null_dim = n_cols - rank_A

ans_3_1 = int(null_dim)
print(f"dim(Null space of A) = {ans_3_1}")

ขนาดของ A: (14, 18)
Rank(A) = 9
dim(Null space of A) = 9


โจทย์ข้อ 3.2: ทำ Least Squares พยากรณ์ spend_k จาก web_visits โดยมี intercept (design matrix = $[1, \text{web\_visits}]$) 
ใช้เฉพาะลูกค้า 891 แถวแรก (แถวที่ 0 ถึง 890) ของไฟล์ biz_customers.csv 
ค่าสัมประสิทธิ์ความชัน $\hat{\beta}_1$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ biz_customers.csv เฉพาะ 891 แถวแรก (แถวที่ 0 ถึง 890)
df_cust = pd.read_csv("biz_customers.csv").iloc[0:891]

# 2. สร้าง design matrix X (ขนาด 891x2) และเวกเตอร์ y
X = np.column_stack([
    np.ones(891, dtype=np.float64),
    df_cust["web_visits"].to_numpy(dtype=np.float64)
])
y = df_cust["spend_k"].to_numpy(dtype=np.float64)

# 3. แก้สมการปกติ (X^T X) beta = X^T y
XTX = X.T @ X
XTy = X.T @ y
beta = np.linalg.solve(XTX, XTy)

# 4. ดึงค่า beta_1 (ความชัน) และปัดเศษทศนิยม 4 ตำแหน่ง
ans_3_2 = round(beta[1], 4)
print(f"beta_0 (intercept) = {beta[0]:.4f}")
print(f"beta_1 (slope of web_visits) = {ans_3_2:.4f}")

beta_0 (intercept) = 1.7059
beta_1 (slope of web_visits) = 0.1211


โจทย์ข้อ 3.3: ให้ $b$ คือค่า spend_k ของลูกค้า 6 คนในแถวที่ 557 ถึง 562 ของไฟล์ biz_customers.csv 
และ $A$ เป็นเมทริกซ์ $6 \times 2$ คือ $[1, \text{tenure\_months}]$ ของ 6 คนเดียวกัน 
ฉายเวกเตอร์ $b$ ลงบน column space ของ $A$ ได้ $p = Pb$ โดย $P = A(A^T A)^{-1} A^T$ 
ความยาว $\|p\|_2$ = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์และเลือกแถวที่ 557 ถึง 562 (6 แถว)
df_cust = pd.read_csv("biz_customers.csv")
sub6 = df_cust.iloc[557:563]

# 2. สร้างเวกเตอร์ b และเมทริกซ์ A (6x2)
b_vec = sub6["spend_k"].to_numpy(dtype=np.float64)
A_mat = np.column_stack([
    np.ones(6, dtype=np.float64),
    sub6["tenure_months"].to_numpy(dtype=np.float64)
])

# 3. สร้าง Projection matrix P = A(A^T A)^{-1} A^T
ATA = A_mat.T @ A_mat
ATA_inv = np.linalg.inv(ATA)
P_mat = A_mat @ ATA_inv @ A_mat.T

# 4. คำนวณ p = P @ b (ผลการฉาย)
p_vec = P_mat @ b_vec

# 5. คำนวณ norm_2 ของ p
norm_p = np.linalg.norm(p_vec)
ans_3_3 = round(norm_p, 3)
print(f"norm_2(p) = {ans_3_3:.3f}")

norm_2(p) = 9.934


โจทย์ข้อ 3.4: กรณีศึกษาธุรกิจสมัครสมาชิกของเครือร้านกาแฟ 
สร้างสมการพยากรณ์ยอดซื้อต่อเดือนจากระยะเวลาที่เป็นลูกค้า (tenure_months) และขนาดตะกร้า (basket_size) 
ด้วย Least Squares (design matrix = $[1, \text{tenure\_months}, \text{basket\_size}]$ มี intercept) โดยใช้ลูกค้า 960 คนแรก 
จากนั้นนำสมการไปพยากรณ์ลูกค้ารายใหม่ซึ่งอยู่แถวที่ 1176 ของไฟล์ (แถวนี้ไม่ได้ใช้ในการ fit) 
ยอดซื้อต่อเดือนที่พยากรณ์ได้ (พันบาท) = ? (ทศนิยม 3 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์และเลือก 960 คนแรกสำหรับ training
df_cust = pd.read_csv("biz_customers.csv")
train = df_cust.iloc[0:960]

# 2. สร้าง design matrix X (ขนาด 960x3) และเวกเตอร์ y
X = np.column_stack([
    np.ones(960, dtype=np.float64),
    train["tenure_months"].to_numpy(dtype=np.float64),
    train["basket_size"].to_numpy(dtype=np.float64)
])
y = train["spend_k"].to_numpy(dtype=np.float64)

# 3. แก้สมการปกติ
beta = np.linalg.solve(X.T @ X, X.T @ y)
print(f"beta = {beta}")

# 4. พยากรณ์ลูกค้าแถวที่ 1176
new_cust = df_cust.iloc[1176]
print(f"ลูกค้าแถว 1176: tenure_months={new_cust['tenure_months']}, basket_size={new_cust['basket_size']}")

y_pred = beta[0] + beta[1]*new_cust["tenure_months"] + beta[2]*new_cust["basket_size"]
ans_3_4 = round(y_pred, 3)
print(f"ยอดซื้อที่พยากรณ์ (พันบาท) = {ans_3_4:.3f}")

beta = [-0.204   0.00478  0.84993]
ลูกค้าแถว 1176: tenure_months=42, basket_size=3.8
ยอดซื้อที่พยากรณ์ (พันบาท) = 3.345


โจทย์ข้อ 3.5: กรณีศึกษาทีมการตลาดดิจิทัล พยากรณ์ยอดขายของแคมเปญทั้ง 240 รายการด้วย Least Squares 
โดยใช้ design matrix 3 คอลัมน์: (1) ค่าคงที่ 1; (2) ตัวบ่งชี้ Q1 (1 ถ้า season=Q1 ไม่เช่นนั้น 0); (3) งบโฆษณาวิทยุ 
ค่าสัมประสิทธิ์ของตัวบ่งชี้ไตรมาส Q1 = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ biz_campaigns.csv (240 แคมเปญ)
df_camp = pd.read_csv("biz_campaigns.csv")

# 2. สร้าง Dummy variable สำหรับ Q1
is_q1 = (df_camp["season"] == "Q1").astype(float).to_numpy()
print(f"จำนวนแคมเปญ Q1: {int(is_q1.sum())} จากทั้งหมด {len(df_camp)}")

# 3. สร้าง design matrix X (240x3) และเวกเตอร์ y
X = np.column_stack([
    np.ones(240, dtype=np.float64),
    is_q1,
    df_camp["radio"].to_numpy(dtype=np.float64)
])
y = df_camp["sales"].to_numpy(dtype=np.float64)

# 4. แก้สมการปกติ
beta = np.linalg.solve(X.T @ X, X.T @ y)

# 5. ดึงค่าสัมประสิทธิ์ Q1 (beta[1])
ans_3_5 = round(beta[1], 4)
print(f"beta_0 (intercept) = {beta[0]:.4f}")
print(f"beta_1 (Q1 indicator) = {ans_3_5:.4f}")
print(f"beta_2 (radio budget) = {beta[2]:.4f}")

จำนวนแคมเปญ Q1: 68 จากทั้งหมด 240
beta_0 (intercept) = 6.9738
beta_1 (Q1 indicator) = -0.9476
beta_2 (radio budget) = 0.2910


โจทย์ข้อ 3.6: สร้าง design matrix $X$ ขนาด $1203 \times 4$ จากลูกค้าทั้ง 1203 คนใน biz_customers.csv 
คือ $[1, \text{age}, \text{tenure\_months}, \text{web\_visits}]$ (ตามลำดับนี้) 
แล้วทำ QR decomposition $X = QR$ ด้วย np.linalg.qr(X) (โมดเริ่มต้น $R$ ขนาด $4 \times 4$) 
ค่าสัมบูรณ์ของสมาชิกแนวทแยงตัวสุดท้าย $|R[3, 3]|$ = ? (ทศนิยม 4 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ biz_customers.csv (ลูกค้าทั้ง 1203 คน)
df_cust = pd.read_csv("biz_customers.csv")

# 2. สร้าง design matrix X (1203x4)
X = np.column_stack([
    np.ones(1203, dtype=np.float64),
    df_cust["age"].to_numpy(dtype=np.float64),
    df_cust["tenure_months"].to_numpy(dtype=np.float64),
    df_cust["web_visits"].to_numpy(dtype=np.float64)
])
print(f"ขนาด X: {X.shape}")

# 3. ทำ QR decomposition X = QR
Q, R = np.linalg.qr(X)
print(f"ขนาด Q: {Q.shape}, R: {R.shape}")

# 4. ดึงค่าสัมบูรณ์ของ R[3,3]
ans_3_6 = round(abs(R[3, 3]), 4)
print(f"abs(R[3, 3]) = {ans_3_6:.4f}")

ขนาด X: (1203, 4)
ขนาด Q: (1203, 4), R: (4, 4)
abs(R[3, 3]) = 115.5478


โจทย์ข้อ 3.7: กรณีศึกษาฝ่ายวางแผนสื่อของเครือร้านค้า 
กำลังจะ fit โมเดลถดถอยพยากรณ์ยอดขายจากงบโฆษณาวิทยุ, โซเชียล, อีเมล ของแคมเปญทั้ง 240 รายการ 
โดย design matrix $X = [1, \text{radio}, \text{social}, \text{email}]$ (ค่าดิบ ไม่ standardize) 
หา condition number (2-norm) ของ $X$ = ? (ทศนิยม 1 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ biz_campaigns.csv
df_camp = pd.read_csv("biz_campaigns.csv")

# 2. สร้าง design matrix X (240x4) ค่าดิบ ไม่ standardize
X = np.column_stack([
    np.ones(240, dtype=np.float64),
    df_camp["radio"].to_numpy(dtype=np.float64),
    df_camp["social"].to_numpy(dtype=np.float64),
    df_camp["email"].to_numpy(dtype=np.float64)
])

# 3. คำนวณ condition number (2-norm) = sigma_max / sigma_min
cond_X = np.linalg.cond(X)

ans_3_7 = round(cond_X, 1)
print(f"Condition number (2-norm) ของ X = {ans_3_7:.1f}")

Condition number (2-norm) ของ X = 150.8


โจทย์ข้อ 3.8: สร้าง design matrix พหุนามดีกรี 3 ของ age จากลูกค้าทั้ง 1203 คนใน biz_customers.csv 
(ค่าดิบ ไม่ย้ายค่ากลาง ไม่ scale) ขนาด $1203 \times 4$ คือ $[1, \text{age}, \text{age}^2, \text{age}^3]$ 
ค่า $\log_{10}$ ของ condition number (2-norm) ของเมทริกซ์นี้ = ? (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์และดึงค่า age
df_cust = pd.read_csv("biz_customers.csv")
age = df_cust["age"].to_numpy(dtype=np.float64)

# 2. สร้าง design matrix พหุนามดีกรี 3 (1203x4)
X = np.column_stack([
    np.ones(1203, dtype=np.float64),  # กำลัง 0
    age,                               # กำลัง 1
    age ** 2,                          # กำลัง 2
    age ** 3                           # กำลัง 3
])

# 3. คำนวณ condition number และ log10
cond_X = np.linalg.cond(X)
log10_cond = np.log10(cond_X)

ans_3_8 = round(log10_cond, 2)
print(f"Condition number = {cond_X:.2e}")
print(f"log10(condition number) = {ans_3_8:.2f}")

Condition number = 2.69e+06
log10(condition number) = 6.43


โจทย์ข้อ 3.9: กรณีศึกษาฝ่ายการตลาดของห้างสรรพสินค้าสงสัยว่ายอดซื้อต่อเดือนไม่ได้เพิ่มตามอายุเป็นเส้นตรง 
จึง fit โมเดล $\text{spend\_k} = \beta_0 + \beta_1 \cdot \text{age} + \beta_2 \cdot \text{age}^2$ ด้วย Least Squares 
เฉพาะลูกค้าที่เป็นสมาชิกสะสมแต้ม (loyalty_member = 1) จำนวน 453 คน 
แล้วหาอายุที่โมเดลนี้ให้ยอดซื้อสูงสุด (จุดยอดของพาราโบลา = $-\beta_1 / (2\beta_2)$) 
อายุที่ทำให้ยอดซื้อสูงสุด = ? ปี (ทศนิยม 2 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. กรองเฉพาะสมาชิกสะสมแต้ม (loyalty_member = 1)
df_cust = pd.read_csv("biz_customers.csv")
loyal = df_cust[df_cust["loyalty_member"] == 1].copy()
print(f"จำนวนสมาชิกสะสมแต้ม: {len(loyal)} คน")

# 2. สร้าง design matrix X (453x3) และเวกเตอร์ y
age_l = loyal["age"].to_numpy(dtype=np.float64)
X = np.column_stack([
    np.ones(len(loyal), dtype=np.float64),
    age_l,
    age_l ** 2
])
y = loyal["spend_k"].to_numpy(dtype=np.float64)

# 3. แก้สมการปกติ
beta = np.linalg.solve(X.T @ X, X.T @ y)
print(f"beta_0 = {beta[0]:.6f}, beta_1 = {beta[1]:.6f}, beta_2 = {beta[2]:.6f}")

# 4. จุดยอดของพาราโบลา: อายุที่ให้ยอดซื้อสูงสุด = -beta1 / (2*beta2)
age_opt = -beta[1] / (2 * beta[2])

ans_3_9 = round(age_opt, 2)
print(f"อายุที่ทำให้ยอดซื้อสูงสุด = {ans_3_9:.2f} ปี")

จำนวนสมาชิกสะสมแต้ม: 453 คน
beta_0 = 0.761706, beta_1 = 0.155220, beta_2 = -0.001766
อายุที่ทำให้ยอดซื้อสูงสุด = 43.95 ปี


โจทย์ข้อ 3.10: กรณีศึกษาฝ่ายบัญชีของเครือร้านค้า ลองประมาณงบโฆษณารวมจากงบ 3 สื่อที่เหลือ 
(งบโฆษณาทีวี, โซเชียล, อีเมล) ด้วย Least Squares 
(design matrix = $[1, \text{tv}, \text{social}, \text{email}]$ มี intercept) 
ผลรวมกำลังสองของส่วนที่เหลือ (RSS) = ? (ทศนิยม 1 ตำแหน่ง)

In [ ]:
import numpy as np
import pandas as pd

# 1. อ่านไฟล์ biz_campaigns.csv (240 แคมเปญ)
df_camp = pd.read_csv("biz_campaigns.csv")

# 2. สร้าง design matrix X (240x4) และ y = total_budget
X = np.column_stack([
    np.ones(240, dtype=np.float64),
    df_camp["tv"].to_numpy(dtype=np.float64),
    df_camp["social"].to_numpy(dtype=np.float64),
    df_camp["email"].to_numpy(dtype=np.float64)
])
y = df_camp["total_budget"].to_numpy(dtype=np.float64)

# 3. แก้สมการปกติ
beta = np.linalg.solve(X.T @ X, X.T @ y)
print(f"beta = {beta}")

# 4. คำนวณ RSS = sum((y - y_hat)^2)
y_hat = X @ beta
residuals = y - y_hat
RSS = np.sum(residuals ** 2)

ans_3_10 = round(RSS, 1)
print(f"RSS (Residual Sum of Squares) = {ans_3_10:.1f}")

beta = [ 2.7453  0.9987  0.9921  1.0108]
RSS (Residual Sum of Squares) = 46556.8


# สรุปทฤษฎีบทและแนวคิดคณิตศาสตร์ (Week 3: Rank, Null Space, Projection, Least Squares และ Condition Number)

---

## 1. Rank และ Rank-Nullity Theorem

* **Rank** ของเมทริกซ์ $A$ ขนาด $m \times n$ คือจำนวนแถว/คอลัมน์ที่เป็นอิสระเชิงเส้น:
  $$\text{rank}(A) = \dim(\text{Col}(A)) = \dim(\text{Row}(A))$$

* **Rank-Nullity Theorem:** สำหรับเมทริกซ์ $A$ ขนาด $m \times n$:
  $$\text{rank}(A) + \dim(\text{Null}(A)) = n \implies \dim(\text{Null}(A)) = n - \text{rank}(A)$$

---

## 2. Null Space

* **Null Space** (เคอร์เนล) ของ $A$: เซตของเวกเตอร์ $x$ ที่ทำให้ $Ax = 0$:
  $$\text{Null}(A) = \{x \in \mathbb{R}^n \mid Ax = 0\}$$
* $\dim(\text{Null}(A)) = 0$: มีแต่คำตอบ $x = 0$ (A full column rank)
* $\dim(\text{Null}(A)) > 0$: มีคำตอบอนันต์

---

## 3. การฉาย (Projection) ลงบน Column Space

* **Projection matrix:**
  $$P = A(A^T A)^{-1} A^T$$
* **เวกเตอร์ฉาย:** $p = Pb$ คือส่วนของ $b$ ที่อยู่ใน column space ของ $A$
* **เวกเตอร์ส่วนที่เหลือ:** $e = b - p$ ตั้งฉากกับ column space ของ $A$ ($A^T e = 0$)
* **สมบัติ:** $P^2 = P$ (Idempotent) และ $P^T = P$ (Symmetric)

---

## 4. Least Squares และสมการปกติ (Normal Equations)

* หา $\hat{\beta}$ ที่ minimize $\|X\beta - y\|^2$:
  $$(X^T X)\hat{\beta} = X^T y \implies \hat{\beta} = (X^T X)^{-1} X^T y$$
* **RSS (Residual Sum of Squares):**
  $$\text{RSS} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \|y - X\hat{\beta}\|^2$$

---

## 5. QR Decomposition

* แยก $X = QR$ โดย $Q$ เป็นเมทริกซ์ orthonormal columns, $R$ เป็นเมทริกซ์สามเหลี่ยมบน
* แก้ Least Squares เสถียรกว่าสมการปกติโดยตรง: $R\hat{\beta} = Q^T y$

---

## 6. Condition Number

* **Condition number (2-norm):** $\kappa(X) = \sigma_{\max} / \sigma_{\min}$
* $\kappa \approx 1$: well-conditioned (เสถียร)
* $\kappa \gg 1$: ill-conditioned (ไม่เสถียร)
* Polynomial features (ค่าดิบ ไม่ centering/scaling) มักทำให้ $\kappa$ สูงมาก
* ใช้ $\log_{10}(\kappa)$ วัดระดับ ill-conditioning: $\log_{10}(\kappa)=6$ สูญเสียความแม่นยำ ~6 หลัก